In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [medmnist]


In [ ]:
"""
===============================================================================
Class-Conditional DDPM with Classifier-Free Guidance on DermaMNIST
===============================================================================
Author      : Portfolio / CV ML Research Project
Task        : Generative Data Augmentation for Imbalanced Medical Datasets
Dataset     : DermaMNIST (MedMNIST v2, 64x64 skin lesion images derived from HAM10000)
Architecture: U-Net with Sinusoidal Positional Embeddings, Class Embeddings, & Self-Attention
Techniques  : Classifier-Free Guidance (CFG), Exponential Moving Average (EMA),
              Automatic Mixed Precision (AMP), Resumable Checkpointing, CSV Logging.
===============================================================================
"""

import argparse
import copy
import csv
import math
import os
import sys
from typing import Tuple, List, Dict, Optional

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.utils as vutils
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

import medmnist
from medmnist import INFO

# -----------------------------------------------------------------------------
# 1. GLOBAL & ENVIRONMENT CONFIGURATION
# -----------------------------------------------------------------------------
NUM_CLASSES = 7  # DermaMNIST classes (0..6)
CLASS_NAMES = [
    "actinic keratoses",
    "basal cell carcinoma",
    "benign keratosis",
    "dermatofibroma",
    "melanoma",
    "melanocytic nevi",
    "vascular lesions",
]


def detect_environment() -> Tuple[str, str]:
    """Detect execution environment (Kaggle vs. Local/Colab) and assign output paths."""
    if os.path.exists("/kaggle/working"):
        out_dir = "/kaggle/working/outputs"
        data_dir = "/kaggle/working/data"
        print("[Env] Detected Kaggle Environment.")
    else:
        out_dir = "./outputs"
        data_dir = "./data"
        print("[Env] Detected Local / Colab Environment.")
    return out_dir, data_dir


# -----------------------------------------------------------------------------
# 2. MODEL ARCHITECTURE (U-Net with Time & Class Conditioning)
# -----------------------------------------------------------------------------
def timestep_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Sinusoidal positional embeddings for diffusion timesteps."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, emb_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.emb = nn.Linear(emb_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.emb(F.silu(emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Attention(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.attn = nn.MultiheadAttention(ch, num_heads=4, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        z = self.norm(x).flatten(2).transpose(1, 2)  # (b, hw, c)
        z, _ = self.attn(z, z, z, need_weights=False)
        return x + z.transpose(1, 2).reshape(b, c, h, w)


class UNet(nn.Module):
    """
    Class-conditioned 64x64 U-Net backbone.
    Class conditioning index `NUM_CLASSES` represents the unconditional state for CFG.
    """
    def __init__(self, base: int = 64, mults: Tuple[int, ...] = (1, 2, 4), emb_dim: int = 256):
        super().__init__()
        self.base = base
        self.time_mlp = nn.Sequential(
            nn.Linear(base, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        self.class_emb = nn.Embedding(NUM_CLASSES + 1, emb_dim)
        self.in_conv = nn.Conv2d(3, base, 3, padding=1)

        chs = [base * m for m in mults]
        self.down_blocks = nn.ModuleList()
        self.downs = nn.ModuleList()
        skip_chs = []
        ch = base

        for i, out in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([ResBlock(ch, out, emb_dim), ResBlock(out, out, emb_dim)]))
            skip_chs.append(out)
            ch = out
            self.downs.append(nn.Conv2d(ch, ch, 3, stride=2, padding=1) if i < len(chs) - 1 else nn.Identity())

        self.mid1 = ResBlock(ch, ch, emb_dim)
        self.mid_attn = Attention(ch)
        self.mid2 = ResBlock(ch, ch, emb_dim)

        self.up_blocks = nn.ModuleList()
        self.ups = nn.ModuleList()
        for i, out in reversed(list(enumerate(chs))):
            self.up_blocks.append(nn.ModuleList([ResBlock(ch + skip_chs[i], out, emb_dim), ResBlock(out, out, emb_dim)]))
            ch = out
            self.ups.append(nn.Upsample(scale_factor=2, mode="nearest") if i > 0 else nn.Identity())

        self.out = nn.Sequential(
            nn.GroupNorm(8, ch),
            nn.SiLU(),
            nn.Conv2d(ch, 3, 3, padding=1)
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        box = timestep_embedding(t, self.base)
        emb = self.time_mlp(box) + self.class_emb(y)
        h = self.in_conv(x)
        skips = []

        for blocks, down in zip(self.down_blocks, self.downs):
            for blk in blocks:
                h = blk(h, emb)
            skips.append(h)
            h = down(h)

        h = self.mid2(self.mid_attn(self.mid1(h, emb)), emb)

        for blocks, up in zip(self.up_blocks, self.ups):
            h = torch.cat([h, skips.pop()], dim=1)
            for blk in blocks:
                h = blk(h, emb)
            h = up(h)

        return self.out(h)


# -----------------------------------------------------------------------------
# 3. DIFFUSION PIPELINE (Gaussian DDPM with Classifier-Free Guidance)
# -----------------------------------------------------------------------------
class Diffusion:
    def __init__(self, T: int = 1000, schedule: str = "cosine", device: torch.device = torch.device("cpu")):
        self.T = T
        self.device = device

        if schedule == "cosine":
            s = 0.008
            steps = torch.arange(T + 1, dtype=torch.float64)
            alphas2 = torch.cos(((steps / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            alphas2 = alphas2 / alphas2[0]
            betas = 1.0 - (alphas2[1:] / alphas2[:-1])
            betas = torch.clip(betas, 0.0001, 0.9999).float()
        else:
            betas = torch.linspace(1e-4, 0.02, T, dtype=torch.float32)

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

        self.betas = betas.to(device)
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod).to(device)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod).to(device)

        # Posterior parameters
        self.posterior_variance = (betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)).to(device)
        self.posterior_mean_coef1 = (betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)).to(device)
        self.posterior_mean_coef2 = ((1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod)).to(device)

    def q_sample(self, x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        at = self.sqrt_alphas_cumprod[t][:, None, None, None]
        ot = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return at * x_0 + ot * noise

    @torch.no_grad()
    def sample(
        self,
        model: nn.Module,
        shape: Tuple[int, ...],
        y: torch.Tensor,
        guidance_scale: float = 2.0
    ) -> torch.Tensor:
        """Ancestral DDPM sampling with Classifier-Free Guidance."""
        n = shape[0]
        x = torch.randn(shape, device=self.device)
        y_uncond = torch.full_like(y, NUM_CLASSES)

        for t_step in reversed(range(self.T)):
            t = torch.full((n,), t_step, device=self.device, dtype=torch.long)

            if guidance_scale > 1.0:
                eps_cond, eps_uncond = torch.chunk(
                    model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([y, y_uncond])), 2, dim=0
                )
                eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
            else:
                eps = model(x, t, y)

            sa = self.sqrt_alphas_cumprod[t][:, None, None, None]
            so = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
            
            # Predict x0 and clamp to standard image range [-1, 1] to prevent numerical divergence
            x0_pred = ((x - so * eps) / sa).clamp(-1.0, 1.0)
            mean = (
                self.posterior_mean_coef1[t][:, None, None, None] * x0_pred
                + self.posterior_mean_coef2[t][:, None, None, None] * x
            )

            if t_step == 0:
                x = mean
            else:
                var = self.posterior_variance[t][:, None, None, None]
                x = mean + torch.sqrt(var) * torch.randn_like(x)

        return torch.clamp(x, -1.0, 1.0)


# -----------------------------------------------------------------------------
# 4. DATA LOADER & DATASET UTILITIES
# -----------------------------------------------------------------------------
def get_dataloader(args) -> DataLoader:
    DataClass = getattr(medmnist, INFO["dermamnist"]["python_class"])
    tfm = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3),
    ])
    ds = DataClass(split="train", transform=tfm, download=True, size=64, root=args.data_dir)
    counts = torch.bincount(torch.tensor(ds.labels.flatten()), minlength=NUM_CLASSES)
    
    print("\n--- DermaMNIST Dataset Class Distribution ---")
    for idx, (name, count) in enumerate(zip(CLASS_NAMES, counts.tolist())):
        print(f"  Class {idx} [{name[:22]:<22}]: {count:5d} samples")
    print("---------------------------------------------\n")

    return DataLoader(
        ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=True if torch.cuda.is_available() else False,
        drop_last=True
    )


# -----------------------------------------------------------------------------
# 5. SYNTHETIC DATASET GENERATION PIPELINE
# -----------------------------------------------------------------------------
@torch.no_grad()
def export_synthetic_dataset(
    model: nn.Module,
    diff: Diffusion,
    output_dir: str,
    samples_per_class: int = 50,
    guidance_scale: float = 2.0,
    batch_size: int = 32
):
    """
    Generates a balanced synthetic dataset for downstream classification/augmentation tasks.
    Saves organized image folders and a master metadata CSV.
    """
    syn_dir = os.path.join(output_dir, "synthetic_dataset")
    os.makedirs(syn_dir, exist_ok=True)
    metadata_path = os.path.join(syn_dir, "metadata.csv")

    print(f"\n[Export] Generating {samples_per_class * NUM_CLASSES} synthetic images for downstream ML tasks...")

    with open(metadata_path, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "class_id", "class_name"])

        for c in range(NUM_CLASSES):
            class_folder = os.path.join(syn_dir, f"class_{c}")
            os.makedirs(class_folder, exist_ok=True)
            num_batches = math.ceil(samples_per_class / batch_size)
            generated_count = 0

            for b in range(num_batches):
                current_b = min(batch_size, samples_per_class - generated_count)
                y_gen = torch.full((current_b,), c, device=diff.device, dtype=torch.long)
                samples = diff.sample(model, (current_b, 3, 64, 64), y_gen, guidance_scale=guidance_scale)
                
                # Rescale [-1, 1] -> [0, 255]
                samples = (samples + 1.0) / 2.0
                samples = (samples * 255).clamp(0, 255).to(torch.uint8).cpu().permute(0, 2, 3, 1).numpy()

                for i in range(current_b):
                    img_name = f"syn_c{c}_{generated_count + i:04d}.png"
                    img_path = os.path.join(class_folder, img_name)
                    Image.fromarray(samples[i]).save(img_path)
                    writer.writerow([os.path.join(f"class_{c}", img_name), c, CLASS_NAMES[c]])

                generated_count += current_b

    print(f"[Export] Synthetic dataset successfully exported to: {syn_dir}\n")


# -----------------------------------------------------------------------------
# 6. TRAINING ENGINE WITH AMP & RESUME CAPABILITY
# -----------------------------------------------------------------------------
def train(args):
    os.makedirs(args.out_dir, exist_ok=True)
    os.makedirs(args.data_dir, exist_ok=True)
    torch.manual_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Compute] Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

    loader = get_dataloader(args)
    model = UNet(base=64, mults=(1, 2, 4)).to(device)
    ema_model = copy.deepcopy(model).eval()
    for p in ema_model.parameters():
        p.requires_grad_(False)

    print(f"[Architecture] Model Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
    diff = Diffusion(T=args.T, schedule=args.schedule, device=device)

    start_epoch = 1
    losses = []
    checkpoint_path = os.path.join(args.out_dir, "checkpoint_latest.pt")
    csv_log_path = os.path.join(args.out_dir, "training_log.csv")

    # Resume Training Checkpoint if existing
    if args.resume and os.path.exists(checkpoint_path):
        print(f"[Resume] Loading checkpoint from {checkpoint_path}...")
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        ema_model.load_state_dict(ckpt["ema_model"])
        opt.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch = ckpt["epoch"] + 1
        losses = ckpt["losses"]
        print(f"[Resume] Resuming training from epoch {start_epoch}")
    else:
        with open(csv_log_path, mode="w", newline="") as f:
            csv.writer(f).writerow(["epoch", "loss"])

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        epoch_loss = 0.0

        for x, y in loader:
            x, y = x.to(device), y.view(-1).long().to(device)

            # Classifier-free guidance: random class label dropout
            mask = torch.rand(x.size(0), device=device) < args.p_uncond
            y_cond = torch.where(mask, torch.full_like(y, NUM_CLASSES), y)
            t = torch.randint(0, args.T, (x.size(0),), device=device)
            noise = torch.randn_like(x)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                x_noisy = diff.q_sample(x, t, noise)
                pred_noise = model(x_noisy, t, y_cond)
                loss = F.mse_loss(pred_noise, noise)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            epoch_loss += loss.item()

            # Exponential Moving Average update
            with torch.no_grad():
                for p, ema_p in zip(model.parameters(), ema_model.parameters()):
                    ema_p.mul_(args.ema).add_(p, alpha=1.0 - args.ema)

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)

        print(f"Epoch [{epoch:03d}/{args.epochs:03d}] | MSE Noise Loss: {avg_loss:.5f}")

        # Append CSV Logs
        with open(csv_log_path, mode="a", newline="") as f:
            csv.writer(f).writerow([epoch, avg_loss])

        # Save Grid Samples and Model Checkpoints
        if epoch % args.sample_every == 0 or epoch == args.epochs:
            # 8 samples per class (Rows: Class 0 to 6)
            y_gen = torch.arange(NUM_CLASSES, device=device).repeat_interleave(8)
            samples = diff.sample(ema_model, (y_gen.size(0), 3, 64, 64), y_gen, guidance_scale=args.guidance)
            
            sample_img_path = os.path.join(args.out_dir, f"samples_epoch_{epoch:03d}.png")
            vutils.save_image(samples.cpu(), sample_img_path, nrow=8, normalize=True, value_range=(-1, 1))

            # Save Latest Checkpoint for Resumability
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "ema_model": ema_model.state_dict(),
                "optimizer": opt.state_dict(),
                "scaler": scaler.state_dict(),
                "losses": losses
            }, checkpoint_path)

            torch.save(ema_model.state_dict(), os.path.join(args.out_dir, "ema_unet_best.pt"))
            print(f"  --> Saved sample grid to '{sample_img_path}' and updated checkpoints.")

    # Save Final High-DPI Loss Curve
    plt.figure(figsize=(7, 4.5))
    plt.plot(range(1, len(losses) + 1), losses, color="#2b5c8f", lw=2, label="DDPM MSE Loss")
    plt.xlabel("Epoch", fontsize=11)
    plt.ylabel("MSE Loss", fontsize=11)
    plt.title("Class-Conditional DDPM Training Loss (DermaMNIST)", fontsize=12, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(args.out_dir, "loss_curve.png"), dpi=300)
    plt.close()

    # Generate Synthetic Dataset Archive
    if args.generate_synthetic:
        export_synthetic_dataset(
            ema_model, diff, args.out_dir,
            samples_per_class=args.synthetic_samples_per_class,
            guidance_scale=args.guidance
        )

    print("\n[Complete] Project execution finished successfully!")


# -----------------------------------------------------------------------------
# 7. ARGUMENT PARSER & ENTRYPOINT (Kaggle / Colab / CLI Compatible)
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    default_out, default_data = detect_environment()

    parser = argparse.ArgumentParser(description="Class-Conditional DDPM on DermaMNIST")
    parser.add_argument("--epochs", type=int, default=200, help="Total training epochs")
    parser.add_argument("--batch_size", type=int, default=64, help="Training batch size")
    parser.add_argument("--lr", type=float, default=2e-4, help="Learning rate")
    parser.add_argument("--T", type=int, default=1000, help="Total diffusion timesteps")
    parser.add_argument("--schedule", choices=["cosine", "linear"], default="cosine", help="Beta schedule")
    parser.add_argument("--p_uncond", type=float, default=0.1, help="Probability of dropping class labels for CFG")
    parser.add_argument("--guidance", type=float, default=2.0, help="Classifier-free guidance scale (w >= 1.0)")
    parser.add_argument("--ema", type=float, default=0.999, help="EMA decay rate")
    parser.add_argument("--sample_every", type=int, default=25, help="Epoch interval to sample grid images")
    parser.add_argument("--workers", type=int, default=2, help="DataLoader workers")
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    parser.add_argument("--data_dir", type=str, default=default_data, help="Directory for dataset cache")
    parser.add_argument("--out_dir", type=str, default=default_out, help="Output directory for checkpoints/logs")
    parser.add_argument("--resume", action="store_true", default=True, help="Automatically resume from latest checkpoint")
    parser.add_argument("--generate_synthetic", action="store_true", default=True, help="Export synthetic dataset after training")
    parser.add_argument("--synthetic_samples_per_class", type=int, default=50, help="Number of synthetic samples per class")

    # parse_known_args() extracts valid flags and ignores Jupyter '-f kernel.json' arguments
    cli_args, _ = parser.parse_known_args()

    train(cli_args)

[Env] Detected Kaggle Environment.
[Compute] Active Device: cuda (Tesla T4)


100%|██████████| 100M/100M [00:37<00:00, 2.70MB/s] 



--- DermaMNIST Dataset Class Distribution ---
  Class 0 [actinic keratoses     ]:   228 samples
  Class 1 [basal cell carcinoma  ]:   359 samples
  Class 2 [benign keratosis      ]:   769 samples
  Class 3 [dermatofibroma        ]:    80 samples
  Class 4 [melanoma              ]:   779 samples
  Class 5 [melanocytic nevi      ]:  4693 samples
  Class 6 [vascular lesions      ]:    99 samples
---------------------------------------------

[Architecture] Model Parameters: 10.52 M
Epoch [001/200] | MSE Noise Loss: 0.21665
